In [1]:
#as we know that the llm have limited context window so we can't provide all the previous messages to the llm every time so for that we use short term memory which only keeps the recent messages in context window and rest of the messages are stored in database or file system or any other persistent storage. So here we will see how to implement short term memory using langgraph.

#here in this we will keep the most recent messges and store the summary of the all the remianing messages 



In [2]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.checkpoint.sqlite import SqliteSaver

from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,RemoveMessage


In [3]:
from dotenv import load_dotenv 
load_dotenv()
import sqlite3

In [4]:
class Summary_schema(MessagesState):
    summary: str 
    
    #by this it will inherit the variable of the messagesstate schema 
    

In [5]:
import os

In [6]:
load_dotenv()


True

In [7]:
# 

In [8]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [9]:
conn = sqlite3.connect(database="demo.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

In [10]:
#lets design the graph for chatting with the llm 

graph=StateGraph(Summary_schema)


In [11]:
def chat_with_llm(state: Summary_schema)-> Summary_schema:
    messages=[]
    if state['summary']:
        messages.append({'role':'system','content':f'The previous conversation summary is : {state['summary']}'})
    response=llm.invoke(state['messages']) 
    messages.extend(state['messages']) #by this it will append after the summary of previous msgs 
    response=llm.invoke(messages)
    
    
    return {'messages':[response]}

In [12]:
#lets also make condition for the creation of the summary of the old messages 
def Should_summarize(state: Summary_schema)-> bool:
    return len(state['messages'])>6

#True and false output
        

In [13]:
#lets make a node that will summarize all the previous messages 
def Summarization(state: Summary_schema)-> Summary_schema:
    #firstly we have to create the summary and then we have to remove the previous messages 
    if state['summary']:
        prompt=(f"Existing summary of messages is {state['summary']}",
                f"Extend the summary using the new above converstaion ") 
    else:
        prompt=(f"Summarize the above new converstaion")
    summary_message=state['messages']+[HumanMessage(content=prompt)]
    summary=llm.invoke(summary_message)

    #now we have to delete the messages of those we have generated the summary 
    message_to_remove=[RemoveMessage(id=m.id) for m in state['messages'][:-2]] #it will 
    # it will avoid the last 2 messages and return all the other messages
    

    # Keep only last 2 messages verbatim
    messages_to_delete = state["messages"][:-2]

    return {
        "summary": summary.content,
        "messages": message_to_remove
    }

In [14]:
#node for the summarizing and deleting the old messages 





In [15]:
graph.add_node('chat',chat_with_llm)
graph.add_node('summarization',Summarization)



graph.add_edge(START,'chat') 
graph.add_conditional_edges('chat',Should_summarize,{True:'summarization',False:'__end__'})
graph.add_edge('summarization',END)

In [16]:
workflow=graph.compile(checkpointer=checkpointer)

In [17]:
config={'configurable':{'thread_id':'32'}} 

In [18]:
# initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
# initial_state={'messages': [HumanMessage(content="What is my name")]}
# initial_state={'messages':[HumanMessage(content='can you tell me a joke')]}
initial_state={'messages': [HumanMessage(content="")],'summary':''}

In [19]:
def run_turn(text: str):
    out = workflow.invoke({"messages": [HumanMessage(content=text)], "summary": ""}, config=config)
    return out

In [20]:
# gives the current version of the state
def show_state():
    snap = workflow.get_state(config)
    vals = snap.values
    print("\n--- STATE ---")
    print("summary:", vals.get("summary", ""))
    print("num_messages:", len(vals.get("messages", [])))
    print("messages:")
    for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content[:80])

In [21]:
run_turn('Quantum Physics')
show_state()


--- STATE ---
summary: 
num_messages: 4
messages:
- HumanMessage : Explain special theory of relativity
- AIMessage : The Special Theory of Relativity, also known as Special Relativity, is a fundame
- HumanMessage : Quantum Physics
- AIMessage : **Quantum Physics**

Quantum physics, also known as quantum mechanics, is a bran


In [22]:
run_turn('How is Albert Einstien related?')
show_state()


--- STATE ---
summary: 
num_messages: 6
messages:
- HumanMessage : Explain special theory of relativity
- AIMessage : The Special Theory of Relativity, also known as Special Relativity, is a fundame
- HumanMessage : Quantum Physics
- AIMessage : **Quantum Physics**

Quantum physics, also known as quantum mechanics, is a bran
- HumanMessage : How is Albert Einstien related?
- AIMessage : **Albert Einstein and Quantum Physics**

Albert Einstein is one of the most infl


In [23]:
run_turn('What are some of Einstien"s famous work')
show_state()


--- STATE ---
summary: Here's a summary of our conversation:

**Conversation Summary**

We discussed the following topics:

1. **Special Theory of Relativity**: We talked about Albert Einstein's theory of special relativity, which challenged the long-held notion of absolute time and space. We discussed the key principles of special relativity, including time dilation, length contraction, and the speed of light as a universal constant.
2. **Quantum Physics**: We explored the principles of quantum physics, including wave-particle duality, uncertainty principle, superposition, and entanglement. We also discussed the mathematical formulation of quantum physics, including the Schrödinger equation.
3. **Albert Einstein's Contributions**: We talked about Einstein's contributions to quantum physics, including his work on the photoelectric effect, quantum theory of specific heat, and the EPR paradox. We also discussed his critique of quantum mechanics and his debate with Niels Bohr.
4. **Einst

In [24]:
run_turn('Explain special theory of relativity')
show_state()


--- STATE ---
summary: 
num_messages: 4
messages:
- HumanMessage : What are some of Einstien"s famous work
- AIMessage : **Albert Einstein's Famous Works**

Albert Einstein is widely regarded as one of
- HumanMessage : Explain special theory of relativity
- AIMessage : **The Special Theory of Relativity**

The special theory of relativity, also kno


In [25]:
# final_state=workflow.invoke(input=initial_state,config=config)

In [26]:
# final_state

In [27]:
#This is simple short term memory which remains for the current execution only

In [28]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence